In [1]:
import socket
import pyaudio
import sys

# ==========================================
# CONFIGURAÇÕES DE REDE E ÁUDIO
# ==========================================
ESP32_IP = '192.168.0.176'
PORT = 7000

# Configurações exatas do I2S do ESP32
FORMAT = pyaudio.paInt16    # 16-bit PCM
CHANNELS = 2                # Estéreo
RATE = 44100                # Taxa de amostragem (44.1 kHz)
CHUNK = 1024                # Tamanho do buffer de leitura

In [2]:
def main():
    print(f"Tentando conectar ao hub de áudio em {ESP32_IP}:{PORT}...")
    
    # 1. Inicia a conexão TCP
    try:
        client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        client_socket.connect((ESP32_IP, PORT))
        print("[SUCESSO] Conectado ao ESP32!")
    except Exception as e:
        print(f"[ERRO] Não foi possível conectar: {e}")
        sys.exit(1)

    # 2. Inicializa o motor de áudio (PyAudio) para saída (Alto-falantes)
    p = pyaudio.PyAudio()
    try:
        stream_out = p.open(format=FORMAT,
                            channels=CHANNELS,
                            rate=RATE,
                            output=True,
                            frames_per_buffer=CHUNK)
        print("[ÁUDIO] Dispositivo de saída de áudio inicializado com sucesso.")
    except Exception as e:
        print(f"[ERRO] Falha ao abrir a placa de som: {e}")
        client_socket.close()
        sys.exit(1)

    print("\n>>> ESCUTANDO LINE-IN DO ESP32 <<<")
    print("Qualquer som injetado no módulo PCM1808 tocará agora nas colunas do PC.")
    print("Pressione Ctrl+C para encerrar.\n")

    # 3. Loop de Receção e Reprodução
    try:
        while True:
            # 1 frame = 2 canais * 2 bytes (16-bit) = 4 bytes
            # Lemos pacotes de exatos CHUNK * 4 bytes
            data = client_socket.recv(CHUNK * 4)
            
            if not data:
                print("\n[AVISO] O ESP32 parou de enviar dados (Conexão encerrada).")
                break
            
            # Escreve o áudio bruto direto nas colunas do PC
            stream_out.write(data)

    except KeyboardInterrupt:
        print("\n[ENCERRANDO] Finalizando a receção de áudio...")
    except Exception as e:
        print(f"\n[ERRO DURANTE STREAMING] Falha na rede ou no áudio: {e}")
    finally:
        # Limpeza correta dos recursos
        stream_out.stop_stream()
        stream_out.close()
        p.terminate()
        client_socket.close()
        print("Programa encerrado.")

if __name__ == "__main__":
    main()

Tentando conectar ao hub de áudio em 192.168.0.176:7000...
[SUCESSO] Conectado ao ESP32!
[ÁUDIO] Dispositivo de saída de áudio inicializado com sucesso.

>>> ESCUTANDO LINE-IN DO ESP32 <<<
Qualquer som injetado no módulo PCM1808 tocará agora nas colunas do PC.
Pressione Ctrl+C para encerrar.


[ENCERRANDO] Finalizando a receção de áudio...
Programa encerrado.
